# Pipeline OCR — Procesamiento de Documentos Reales
**Notebook Local — Laptop**

Sube cualquier documento real (PNG, JPG, PDF) y el pipeline:
1. **Fase 3A** — Detecta y elimina sellos (clasificación cromática + inpainting)
2. **Fase 4** — Extrae texto con EasyOCR + corrección ortográfica
3. **Visualiza** — Muestra evidencias de detección con color-coding por confianza
4. **Métricas** — Dashboard con histograma, tabla y calificación de calidad

---
**Instrucciones**: Ejecuta las celdas de arriba hacia abajo (`Shift+Enter`).  
La Celda 3 te mostrará un botón para cargar tu documento.

## Celda 1 — Setup: Rutas y Dependencias

In [ ]:
import sys
import os
import subprocess
from pathlib import Path
from datetime import datetime

# === Detectar raiz del repositorio ===
NOTEBOOK_DIR = Path(os.getcwd())
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# === Carpeta de salida (con timestamp para no pisar resultados anteriores) ===
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = REPO_ROOT / "output" / "demo_real" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"[OK] Repositorio : {REPO_ROOT}")
print(f"[OK] Salida      : {OUTPUT_ROOT}")
print(f"[OK] Python      : {sys.version.split()[0]}")

# === Verificar dependencias criticas ===
missing = []
for pkg, imp in [("opencv","cv2"),("numpy","numpy"),("matplotlib","matplotlib"),
                  ("easyocr","easyocr"),("ipywidgets","ipywidgets")]:
    try:
        __import__(imp)
        print(f"[OK] {pkg}")
    except ImportError:
        print(f"[!!] {pkg} — falta  →  pip install {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstala los paquetes faltantes y reinicia el kernel.")
else:
    print("\n[OK] Todas las dependencias listas.")

## Celda 2 — Importar Funciones del Pipeline

In [ ]:
from pipeline import (
    run_phase_3a,
    run_phase_4,
    consolidate_ocr_results,
)

print("[OK] Funciones del pipeline importadas")
print("  run_phase_3a()          → Eliminacion de sello")
print("  run_phase_4()           → OCR + correccion ortografica")
print("  consolidate_ocr_results() → Texto plano + metricas JSON")

## Celda 3 — Cargar Documento Real

Haz click en **"Seleccionar archivo"** para subir tu documento (PNG, JPG o PDF).  
Si prefieres usar una ruta directa, cambia `USE_WIDGET = False` y edita `RUTA_DIRECTA`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import cv2
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ===========================================================================
# CONFIGURACION
# ===========================================================================
USE_WIDGET = True   # True = widget de carga | False = usar ruta directa
RUTA_DIRECTA = r""  # Solo si USE_WIDGET = False: pon aqui la ruta a tu imagen
# ===========================================================================

INPUT_IMAGE = None  # Se asigna debajo

def mostrar_imagen(path, titulo="Documento cargado"):
    """Muestra la imagen con info de dimensiones."""
    img = cv2.imread(str(path))
    if img is None:
        print(f"[ERROR] No se pudo leer: {path}")
        return False
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    size_kb = Path(path).stat().st_size / 1024

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(img_rgb)
    ax.set_title(f"{titulo}\n{Path(path).name}  |  {w}x{h} px  |  {size_kb:.1f} KB",
                 fontsize=12, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return True


if not USE_WIDGET:
    # --- Modo ruta directa ---
    if not RUTA_DIRECTA:
        print("[ERROR] Pon la ruta en RUTA_DIRECTA y vuelve a ejecutar.")
    elif not Path(RUTA_DIRECTA).exists():
        print(f"[ERROR] Archivo no encontrado: {RUTA_DIRECTA}")
    else:
        INPUT_IMAGE = RUTA_DIRECTA
        print(f"[OK] Imagen cargada desde ruta directa: {INPUT_IMAGE}")
        mostrar_imagen(INPUT_IMAGE, "Documento (ruta directa)")
else:
    # --- Modo widget ---
    upload_btn = widgets.FileUpload(
        accept="image/png,image/jpeg,image/jpg,image/bmp,image/tiff",
        multiple=False,
        description="Seleccionar archivo",
        button_style="primary",
        layout=widgets.Layout(width="220px")
    )
    status_out = widgets.Output()

    def on_upload(change):
        global INPUT_IMAGE
        with status_out:
            clear_output(wait=True)
            if not upload_btn.value:
                return
            # Guardar archivo subido en carpeta de salida
            uploaded_file = list(upload_btn.value.values())[0]
            fname = uploaded_file["metadata"]["name"]
            dest = OUTPUT_ROOT / "input" / fname
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(dest, "wb") as f:
                f.write(uploaded_file["content"])
            INPUT_IMAGE = str(dest)
            print(f"[OK] Archivo recibido: {fname}")
            print(f"     Guardado en: {dest}")
            mostrar_imagen(INPUT_IMAGE, "Documento cargado")

    upload_btn.observe(on_upload, names="value")

    print("Haz click en el boton para seleccionar tu documento:")
    display(widgets.VBox([
        upload_btn,
        widgets.HTML("<small style='color:gray'>Formatos aceptados: PNG, JPG, BMP, TIFF</small>"),
        status_out
    ]))

## Celda 4 — Fase 3A: Eliminacion de Sello + Cuadricula de Evidencias

In [ ]:
import time

if not INPUT_IMAGE or not Path(INPUT_IMAGE).exists():
    print("[ERROR] Primero carga una imagen en la Celda 3.")
else:
    PHASE_3A_DIR = str(OUTPUT_ROOT / "phase_3a")

    print("=" * 65)
    print(" FASE 3A — Deteccion y Eliminacion de Sello")
    print("=" * 65)
    print(f" Imagen entrada : {Path(INPUT_IMAGE).name}")
    print(f" Metodo         : Hibrido (TELEA + Navier-Stokes)")
    print(f" Block size     : 20 px")
    print("=" * 65)
    print(" Procesando...\n")

    t0 = time.time()
    result_3a = run_phase_3a(
        input_image_path=INPUT_IMAGE,
        output_dir=PHASE_3A_DIR,
        block_size=20,
        inpainting_method="hybrid",
        save_intermediate=True,
    )
    elapsed_3a = time.time() - t0

    if result_3a["success"]:
        CLEANED_IMAGE = result_3a["output_image"]
        px = result_3a.get("pixels_inpainted", "N/A")
        print(f" [OK] Completado en {elapsed_3a:.1f}s")
        print(f"      Pixeles inpainted : {px}")
        print(f"      Imagen limpia     : {Path(CLEANED_IMAGE).name}")

        # --- Cuadricula de evidencias ---
        out_path = Path(PHASE_3A_DIR)
        imgs_inter = sorted(out_path.glob("0*.png"))
        etiquetas_map = {
            "01": "Original",
            "02": "Clasificacion cromatica",
            "03": "H-Sweep",
            "04": "Block Grid Overlay",
            "05": "Mascara final",
            "06": "Antes vs Despues",
        }

        evidencias, etiquetas = [], []
        for p in imgs_inter[:6]:
            img = cv2.imread(str(p))
            if img is not None:
                evidencias.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                etiquetas.append(etiquetas_map.get(p.stem[:2], p.stem))

        if len(evidencias) >= 2:
            cols = min(3, len(evidencias))
            rows = -(-len(evidencias) // cols)  # ceil division
            fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4.5))
            axes = np.array(axes).flatten()
            for idx, (img, lbl) in enumerate(zip(evidencias, etiquetas)):
                axes[idx].imshow(img)
                axes[idx].set_title(lbl, fontsize=11, fontweight="bold")
                axes[idx].axis("off")
            for idx in range(len(evidencias), len(axes)):
                axes[idx].axis("off")
            plt.suptitle("Fase 3A — Cuadricula de Evidencias",
                         fontsize=14, fontweight="bold", y=1.01)
            plt.tight_layout()
            plt.show()
        else:
            # Fallback: antes/despues simple
            orig = cv2.cvtColor(cv2.imread(INPUT_IMAGE), cv2.COLOR_BGR2RGB)
            clean = cv2.cvtColor(cv2.imread(CLEANED_IMAGE), cv2.COLOR_BGR2RGB)
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            axes[0].imshow(orig);  axes[0].set_title("Original", fontweight="bold"); axes[0].axis("off")
            axes[1].imshow(clean); axes[1].set_title("Sin sello", fontweight="bold"); axes[1].axis("off")
            plt.tight_layout(); plt.show()
    else:
        err = result_3a.get("error", "error desconocido")
        print(f" [!!] Fase 3A fallo: {err}")
        print("      Continuando con la imagen original para OCR...")
        CLEANED_IMAGE = INPUT_IMAGE

## Celda 5 — Fase 4: OCR + Extraccion de Texto

In [ ]:
import json

PHASE_4_DIR = str(OUTPUT_ROOT / "phase_4")

print("=" * 65)
print(" FASE 4 — OCR + Correccion Ortografica + NER")
print("=" * 65)
print(" Motor   : EasyOCR (espanol)")
print(" Umbral  : confianza >= 0.6")
print(" Correc. : Levenshtein distancia <= 2")
print(" NER     : spaCy es_core_news_sm (si disponible)")
print("=" * 65)
print(" NOTA: Primera ejecucion descarga modelo (~100 MB, ~5 min)")
print(" Ejecuciones siguientes son mucho mas rapidas.\n")

t0 = time.time()
result_4 = run_phase_4(
    input_image_path=CLEANED_IMAGE,
    output_dir=PHASE_4_DIR,
    language="es",
    confidence_threshold=0.6,
    levenshtein_threshold=2,
)
elapsed_4 = time.time() - t0

OCR_JSON = None

if result_4["success"]:
    OCR_JSON = result_4["output_json"]
    bloques_n = result_4.get("text_blocks", 0)
    conf_avg  = result_4.get("confidence", 0)
    corregidas = result_4.get("words_corrected", 0)
    chars_n   = result_4.get("text_length", 0)

    print(f" [OK] Completado en {elapsed_4:.1f}s")
    print(f"      Bloques detectados : {bloques_n}")
    print(f"      Confianza promedio : {conf_avg:.2%}")
    print(f"      Palabras corregidas: {corregidas}")
    print(f"      Caracteres totales : {chars_n:,}")

    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)

    texto = ocr_data.get("texto_completo", "")
    preview = texto[:1000]
    print("\n" + "-" * 65)
    print(" TEXTO EXTRAIDO (primeros 1000 caracteres)")
    print("-" * 65)
    print(preview)
    if len(texto) > 1000:
        print(f"\n... [{len(texto)-1000} caracteres mas]")
    print("-" * 65)
else:
    print(f" [ERROR] Fase 4 fallo: {result_4.get('error', 'desconocido')}")

## Celda 6 — Visualizacion de Detecciones OCR con Color-Coding

In [ ]:
if not OCR_JSON or not Path(OCR_JSON).exists():
    print("[INFO] Sin salida OCR — ejecuta la Celda 5 primero.")
else:
    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)

    bloques = ocr_data.get("bloques", [])
    img_bgr = cv2.imread(CLEANED_IMAGE)
    img_vis = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()

    counts = {"alta": 0, "media": 0, "baja": 0}

    for bloque in bloques:
        coords    = bloque.get("coordenadas", [])
        confianza = bloque.get("confianza", 0)

        if len(coords) != 4:
            continue

        # Color segun confianza
        if confianza >= 0.8:
            color, cat = (34, 197, 94), "alta"    # verde
        elif confianza >= 0.5:
            color, cat = (251, 191, 36), "media"  # amarillo
        else:
            color, cat = (239, 68, 68), "baja"    # rojo

        counts[cat] += 1

        pts = np.array(coords, dtype=np.int32).reshape((-1, 1, 2))
        cv2.polylines(img_vis, [pts], True, color, 2)

        # Etiqueta de confianza
        x0, y0 = int(coords[0][0]), max(int(coords[0][1]) - 5, 12)
        cv2.putText(img_vis, f"{confianza:.0%}",
                    (x0, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

    # === Leyenda en esquina superior izquierda ===
    leyenda = [
        ((34, 197, 94),  f"Alta (>=80%)  : {counts['alta']} bloques"),
        ((251, 191, 36), f"Media (50-80%): {counts['media']} bloques"),
        ((239, 68, 68),  f"Baja (<50%)   : {counts['baja']} bloques"),
    ]
    pad = 12
    for i, (col, txt) in enumerate(leyenda):
        y = pad + 28 + i * 26
        cv2.rectangle(img_vis, (pad, y - 12), (pad + 16, y + 4), col, -1)
        cv2.putText(img_vis, txt, (pad + 22, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (30, 30, 30), 1, cv2.LINE_AA)

    total = len(bloques)

    fig, ax = plt.subplots(figsize=(15, 12))
    ax.imshow(img_vis)
    ax.set_title(
        f"Detecciones OCR — {total} bloques  |  "
        f"Alta: {counts['alta']}  Media: {counts['media']}  Baja: {counts['baja']}",
        fontsize=13, fontweight="bold"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"\n Bloques totales     : {total}")
    print(f" Alta confianza (>=80%): {counts['alta']}  ({counts['alta']/max(total,1)*100:.1f}%)")
    print(f" Media confianza      : {counts['media']}  ({counts['media']/max(total,1)*100:.1f}%)")
    print(f" Baja confianza (<50%): {counts['baja']}  ({counts['baja']/max(total,1)*100:.1f}%)")

## Celda 7 — Consolidar Resultados + Dashboard de Metricas

In [ ]:
if not OCR_JSON or not Path(OCR_JSON).exists():
    print("[INFO] Sin salida OCR — ejecuta la Celda 5 primero.")
else:
    CONS_DIR = str(OUTPUT_ROOT / "consolidated")

    print("Consolidando resultados...")
    metrics = consolidate_ocr_results(ocr_json_path=OCR_JSON, output_dir=CONS_DIR)

    txt_file  = Path(CONS_DIR) / "ocr_extracted_text.txt"
    json_file = Path(CONS_DIR) / "ocr_metrics.json"
    print(f"[OK] {txt_file.name}  ({txt_file.stat().st_size/1024:.1f} KB)")
    print(f"[OK] {json_file.name} ({json_file.stat().st_size/1024:.1f} KB)")

    tm = metrics.get("text_metrics", {})
    cm = metrics.get("ocr_confidence_metrics", {})
    pm = metrics.get("post_processing_metrics", {})

    with open(OCR_JSON, encoding="utf-8") as f:
        ocr_data = json.load(f)
    confs = [b.get("confianza", 0) for b in ocr_data.get("bloques", [])]

    avg_conf = cm.get("average_percent", 0) / 100
    if avg_conf >= 0.85:
        quality, qcolor = "EXCELENTE", "#16a34a"
    elif avg_conf >= 0.75:
        quality, qcolor = "MUY BUENO", "#2563eb"
    elif avg_conf >= 0.60:
        quality, qcolor = "BUENO",     "#d97706"
    else:
        quality, qcolor = "MEJORABLE", "#dc2626"

    # ===== DASHBOARD 4 PANELES =====
    fig = plt.figure(figsize=(16, 10))
    gs  = fig.add_gridspec(3, 2, hspace=0.38, wspace=0.28)

    # Panel 1 — Histograma de confianza
    ax1 = fig.add_subplot(gs[0, 0])
    if confs:
        bin_edges   = [0, 0.4, 0.6, 0.8, 0.9, 1.01]
        bin_colors  = ["#ef4444","#f97316","#facc15","#4ade80","#16a34a"]
        hist, edges = np.histogram(confs, bins=bin_edges)
        for i, (h, c) in enumerate(zip(hist, bin_colors)):
            ax1.bar(i, h, color=c, edgecolor="white", width=0.7)
        ax1.set_xticks(range(5))
        ax1.set_xticklabels(["<40%","40-60%","60-80%","80-90%",">90%"], fontsize=8)
        ax1.set_ylabel("Bloques", fontweight="bold")
        ax1.set_title("Distribucion de Confianza", fontweight="bold")
        ax1.grid(axis="y", alpha=0.3)
        for i, h in enumerate(hist):
            if h: ax1.text(i, h + 0.3, str(h), ha="center", fontsize=9, fontweight="bold")

    # Panel 2 — Pie alta/media/baja
    ax2 = fig.add_subplot(gs[0, 1])
    alta  = sum(1 for c in confs if c >= 0.8)
    media = sum(1 for c in confs if 0.5 <= c < 0.8)
    baja  = sum(1 for c in confs if c < 0.5)
    sizes = [alta, media, baja]
    if sum(sizes) > 0:
        pielabels = [f"Alta\n{alta}", f"Media\n{media}", f"Baja\n{baja}"]
        ax2.pie(sizes, labels=pielabels, colors=["#16a34a","#d97706","#dc2626"],
                autopct="%1.0f%%", startangle=90, textprops={"fontsize": 9})
        ax2.set_title("Alta / Media / Baja confianza", fontweight="bold")

    # Panel 3 — Tabla de metricas
    ax3 = fig.add_subplot(gs[1, :])
    ax3.axis("off")
    rows = [
        ["Caracteres totales",       f"{tm.get('total_characters',0):,}"],
        ["Palabras totales",         f"{tm.get('total_words',0):,}"],
        ["Palabras unicas",          f"{tm.get('unique_words',0):,}"],
        ["Bloques de texto",         f"{tm.get('total_blocks',0)}"],
        ["Confianza promedio",        f"{cm.get('average_percent',0):.2f}%"],
        ["Bloques alta confianza",    f"{cm.get('blocks_with_high_confidence',0)}"],
        ["Palabras corregidas",       f"{pm.get('words_corrected',0)}"],
        ["Entidades detectadas (NER)",f"{pm.get('entities_detected',0)}"],
    ]
    tbl = ax3.table(cellText=rows, colLabels=["Metrica", "Valor"],
                    cellLoc="left", loc="center", colWidths=[0.58, 0.28])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 2.0)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#1e40af"); cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#f0f9ff")
        else:
            cell.set_facecolor("#ffffff")

    # Panel 4 — Calificacion de calidad
    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis("off")
    ax4.text(
        0.5, 0.5,
        f"Calidad OCR: {quality}    ({cm.get('average_percent',0):.2f}% confianza promedio)",
        fontsize=17, fontweight="bold", ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.9", facecolor=qcolor,
                  alpha=0.15, edgecolor=qcolor, linewidth=3),
        color=qcolor,
        transform=ax4.transAxes,
    )

    plt.suptitle("Dashboard de Metricas OCR", fontsize=14, fontweight="bold")
    plt.show()

    print(f"\n Calidad    : {quality}")
    print(f" Confianza  : {cm.get('average_percent',0):.2f}%")
    print(f" Caracteres : {tm.get('total_characters',0):,}")
    print(f" Palabras   : {tm.get('total_words',0):,}")
    print(f" Corregidas : {pm.get('words_corrected',0)}")

## Celda 8 — Evaluacion CER/WER (Opcional, requiere ground truth)

Solo si tienes el texto correcto del documento para comparar.  
Pon `HAS_GT = True` y la ruta al archivo `.txt` con el texto real.

In [ ]:
HAS_GT = False
GT_FILE = r""  # Ruta al archivo .txt con el texto correcto del documento

if not HAS_GT:
    print("[INFO] Evaluacion desactivada.")
    print("       Para activar: pon HAS_GT = True y la ruta en GT_FILE")
else:
    if not GT_FILE or not Path(GT_FILE).exists():
        print(f"[ERROR] Archivo ground truth no encontrado: {GT_FILE}")
    else:
        txt_file = Path(CONS_DIR) / "ocr_extracted_text.txt"
        with open(txt_file, encoding="utf-8") as f:
            ocr_text = " ".join(
                l for l in f.read().splitlines()
                if l.strip() and not l.startswith("=") and not l.startswith("Total")
            ).strip()
        with open(GT_FILE, encoding="utf-8") as f:
            gt_text = f.read().strip()

        try:
            from src.phase_5.levenshtein_calculator import LevenshteinCalculator
            lev = LevenshteinCalculator()

            ref_ch, hyp_ch = list(gt_text.lower()), list(ocr_text.lower())
            cer = lev.calculate(ref_ch, hyp_ch) / max(len(ref_ch), 1)

            ref_w, hyp_w = gt_text.lower().split(), ocr_text.lower().split()
            wer = lev.calculate(ref_w, hyp_w) / max(len(ref_w), 1)

            print("=" * 50)
            print(" RESULTADOS DE EVALUACION")
            print("=" * 50)
            print(f" CER (tasa error char) : {cer*100:6.2f}%")
            print(f" WER (tasa error word) : {wer*100:6.2f}%")
            print(f" Precision caracter    : {(1-cer)*100:6.2f}%")
            print(f" Precision palabra     : {(1-wer)*100:6.2f}%")

            if cer < 0.05:   q = "EXCELENTE"
            elif cer < 0.10: q = "MUY BUENO"
            elif cer < 0.20: q = "BUENO"
            elif cer < 0.50: q = "REGULAR"
            else:             q = "MEJORABLE"
            print(f" Calificacion          : {q}")

        except ImportError:
            print("[INFO] LevenshteinCalculator no disponible.")

## Celda 9 — Resumen Final y Ubicacion de Archivos

In [ ]:
print("=" * 70)
print(" RESUMEN FINAL")
print("=" * 70)

if INPUT_IMAGE:
    print(f"\n Documento procesado : {Path(INPUT_IMAGE).name}")

print(f"\n Carpeta de resultados:")
print(f"   {OUTPUT_ROOT}")

archivos = [
    (OUTPUT_ROOT / "phase_3a",                         "Imagen limpia (sin sello)"),
    (OUTPUT_ROOT / "phase_4",                          "JSON con bloques OCR + coords + confianza"),
    (OUTPUT_ROOT / "consolidated" / "ocr_extracted_text.txt", "Texto extraido (plano)"),
    (OUTPUT_ROOT / "consolidated" / "ocr_metrics.json",       "Metricas JSON detalladas"),
]

print()
for ruta, desc in archivos:
    existe = ruta.exists()
    estado = "[OK]" if existe else "[ ] "
    if existe and ruta.is_file():
        kb = ruta.stat().st_size / 1024
        print(f" {estado} {ruta.name:40} {kb:6.1f} KB  — {desc}")
    elif existe and ruta.is_dir():
        n = len(list(ruta.glob("*")))
        print(f" {estado} {ruta.name:40} {n:5} archivos — {desc}")
    else:
        print(f" {estado} {ruta.name:40}          (aun no generado)")

print()
print(" Para ver el texto extraido:")
txt_path = OUTPUT_ROOT / "consolidated" / "ocr_extracted_text.txt"
if txt_path.exists():
    with open(txt_path, encoding="utf-8") as f:
        preview = f.read(400)
    print("-" * 70)
    print(preview)
    print("-" * 70)
print("\n Pipeline completado.")